In [72]:
get_ipython().system('rm -rf sample_data')

In [73]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import torch.nn as nn
from PIL import Image
from pathlib import Path
import torch.optim as optim

设置dataset

In [74]:
from pathlib import Path
from PIL import Image

img_dir = Path("/content/img")
mask_dir = Path("/content/mask")

img_gr_dir = Path("/content/img_gr")
mask_gr_dir = Path("/content/mask_gr")

img_gr_dir.mkdir(exist_ok=True)
mask_gr_dir.mkdir(exist_ok=True)

for img_path in img_dir.iterdir():

    if img_path.suffix.lower() not in [".jpg",".jpeg",".png"]:
        continue

    mask_path = mask_dir / (img_path.stem + ".png")

    img = Image.open(img_path).convert("L")
    mask = Image.open(mask_path).convert("L")

    img.save(img_gr_dir / img_path.name)
    mask.save(mask_gr_dir / mask_path.name)

In [75]:
batch_size=4
epochs = 50
lr = 0.005

In [76]:

class MyDataset(Dataset):
  def __init__(self,img_gr_dir,mask_gr_dir):
    self.mask_dir=Path(mask_gr_dir)
    self.img_dir=Path(img_gr_dir)
    self.transform=transforms.ToTensor()
    self.img_paths = sorted(list(self.img_dir.glob("*.*")))
    self.mask_paths = sorted(list(self.mask_dir.glob("*.*")))

  def __len__(self):
    return len(self.img_paths)

  def __getitem__(self,idx):
    mask_path=self.mask_paths[idx]
    img_path=self.img_paths[idx]

    img = Image.open(img_path)
    mask = Image.open(mask_path)

    img_tensor=self.transform(img)
    mask_tensor=self.transform(mask)

    return img_tensor,mask_tensor


full_dataset=MyDataset(img_gr_dir,mask_gr_dir)
print("totla:", len(full_dataset))

train_size=int(0.8*len(full_dataset))
val_size=len(full_dataset)-train_size

generator=torch.Generator().manual_seed(42)

train_dataset,val_dataset=random_split(
  full_dataset,
  [train_size,val_size],
  generator=generator
 )
train_loader=DataLoader(
     train_dataset,
     batch_size=batch_size,
     shuffle=True
 )

val_loader=DataLoader(
     val_dataset,
     batch_size=batch_size,
     shuffle=False
 )

images, masks=next(iter(train_loader))
print("images.shape=",images.shape)
print("masks.shape=",masks.shape)
print("image min/max =", images.min().item(), images.max().item())
print("mask min/max =", masks.min().item(), masks.max().item())
print(torch.unique(masks))

totla: 61
images.shape= torch.Size([4, 1, 512, 512])
masks.shape= torch.Size([4, 1, 512, 512])
image min/max = 0.0 0.9333333373069763
mask min/max = 0.0 1.0
tensor([0., 1.])


CNN model

In [77]:
class SimpleUnet(nn.Module):
  def __init__(self,in_channels=1,out_channels=1):
    super().__init__()

    self.c1a=nn.Conv2d(in_channels,32,3,padding=1);
    self.c1b=nn.Conv2d(32,32,3,padding=1);
    self.p1=nn.MaxPool2d(2)

    self.c2a=nn.Conv2d(32,64,3,padding=1);
    self.c2b=nn.Conv2d(64,64,3,padding=1);
    self.p2=nn.MaxPool2d(2)

    self.c3a=nn.Conv2d(64,128,3,padding=1);
    self.c3b=nn.Conv2d(128,128,3,padding=1);
    self.p3=nn.MaxPool2d(2)

    self.c4a=nn.Conv2d(128,256,3,padding=1);
    self.c4b=nn.Conv2d(256,256,3,padding=1);

    self.up1=nn.Upsample(scale_factor=2,mode="nearest");
    self.u1a=nn.Conv2d(256+128,128,3,padding=1);
    self.u1b=nn.Conv2d(128,128,3,padding=1);

    self.up2=nn.Upsample(scale_factor=2,mode="nearest");
    self.u2a=nn.Conv2d(128+64,64,3,padding=1);
    self.u2b=nn.Conv2d (64,64,3,padding=1);

    self.up3=nn.Upsample(scale_factor=2,mode="nearest");
    self.u3a=nn.Conv2d(64+32,32,3,padding=1);
    self.u3b=nn.Conv2d(32,32,3,padding=1);

    self.out=nn.Conv2d(32,out_channels,1);
    self.act=nn.ReLU(inplace=True);
    self.final=nn.Sigmoid() if out_channels==1 else nn.Softmax(dim=1)

  def forward(self,x):
    x1=self.act(self.c1a(x));
    x1=self.act(self.c1b(x1));
    p1=self.p1(x1);

    x2=self.act(self.c2a(p1));
    x2=self.act(self.c2b(x2));
    p2=self.p2(x2);

    x3=self.act(self.c3a(p2));
    x3=self.act(self.c3b(x3));
    p3=self.p3(x3);

    b=self.act(self.c4a(p3));
    b=self.act(self.c4b(b));

    y=self.up1(b)
    y=torch.cat([y,x3],dim=1)
    y=self.act(self.u1a(y))
    y=self.act(self.u1b(y))

    y=self.up2(y)
    y=torch.cat([y,x2],dim=1)
    y=self.act(self.u2a(y))
    y=self.act(self.u2b(y))

    y=self.up3(y)
    y=torch.cat([y,x1],dim=1)
    y=self.act(self.u3a(y))
    y=self.act(self.u3b(y))

    return self.final(self.out(y))

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")





创建模型，创建优化器，创建损失函数，创建自动学习率

In [78]:
segmentation_model=SimpleUnet(1,1).to(device)
optimizer=optim.Adam(segmentation_model.parameters(),lr=lr)
criterion = nn.BCELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.2,
    patience=1
)

开始训练


In [79]:
train_losses=[]
val_losses=[]

for epoch in range(epochs):
  #切换到训练模式
  segmentation_model.train()
  #初始损失
  train_loss=0.0


  for img,mask in train_loader:

    img = img.to(device)
    mask = mask.to(device)

    #输入input进模型，得到预测output
    preds=segmentation_model(img)
    #损失函数计算误差
    loss=criterion(preds,mask)
    #每次算grad之前把上次的grad清零
    optimizer.zero_grad()
    #误差反向传播
    loss.backward()
    #算出本次梯度
    optimizer.step()
    #本轮所有图片的误差合
    train_loss+=loss.item()

  #计算每一轮的平均误差，并记录在误差列表
  avg_train_loss=train_loss/len(train_loader)

  segmentation_model.eval()
  val_loss = 0.0
  with torch.no_grad():
    for img,mask in val_loader:
      img = img.to(device)
      mask = mask.to(device)
      preds=segmentation_model(img)
      loss=criterion(preds,mask)
      val_loss+=loss.item()
  avg_val_loss=val_loss/len(val_loader)

  train_losses.append(avg_train_loss)
  val_losses.append(avg_val_loss)
  scheduler.step(avg_val_loss)

  print(f"Epoch{epoch+1}/{epochs}, train loss:{avg_train_loss:6f},val loss:{avg_val_loss:6f}，lr:{optimizer.param_groups[0]['lr']:.6f}")





Epoch1/50, train loss:1.058810,val loss:1.173878，lr:0.005000
Epoch2/50, train loss:1.187627,val loss:1.173878，lr:0.005000
Epoch3/50, train loss:1.187627,val loss:1.173878，lr:0.001000
Epoch4/50, train loss:1.187627,val loss:1.173878，lr:0.001000
Epoch5/50, train loss:1.187627,val loss:1.173878，lr:0.000200
Epoch6/50, train loss:1.187627,val loss:1.173878，lr:0.000200


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime

result_name=f"batch size:{batch_size}, eopchs:{epochs}, lr:{lr}"
exp_dir=Path("/content/result")
exp_dir.mkdir(parents=True,exist_ok=True)
time_str = datetime.now().strftime("%Y%m%d_%H%M%S")

plt.figure()

plt.plot(train_losses,label="train loss")
plt.plot(val_losses,label="val loss")

plt.xlabel("Epoch")
plt.ylabel("loss")
plt.title("training Curve")
plt.legend()

result_path=exp_dir/f"{result_name}{time_str}.png"
plt.savefig(result_path)
plt.show()

print("图已保存:", result_path)